<a href="https://colab.research.google.com/github/Shaxzod1991/ABC_Analysis/blob/main/ABC_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [141]:
import pandas as pd
import numpy as np
!pip install duckdb
import duckdb
pd.set_option('display.float_format', '{:,.2f}'.format)

In [142]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ***Загрузка файла формата CSV***

In [143]:
df_test = pd.read_csv('/content/test/_Тестовое задание 1 (выгрузка).csv', encoding='cp1251', sep='\\t', engine='python')

for i in df_test.columns[4:]:
    df_test[i] = df_test[i].astype(str).str.replace(',', '.')
    df_test[i] = df_test[i].astype(str).str.replace('"', '')
    df_test[i] = pd.to_numeric(df_test[i], errors='coerce')

# ***Обработка Исходника***

In [144]:
query_sql = """

    SELECT
        *,
        "Сумма продаж, тыс.руб." / "Кол проданного товара, шт" AS "Цена за ед., тыс.руб",
        "Себестоимость продаж, тыс.руб." / "Кол проданного товара, шт" AS "Себес. за ед., тыс. руб.",
        "Сумма продаж, тыс.руб." / "Кол проданного товара, шт" - "Себестоимость продаж, тыс.руб." / "Кол проданного товара, шт" AS "Маржа за ед., тыс. руб",
        SUM("Сумма продаж, тыс.руб." - "Себестоимость продаж, тыс.руб.") OVER(PARTITION BY "Номенклатура", "Месяц") AS "Маржа всего за прод, тыс. руб.",
        SUM("Сумма продаж, тыс.руб." - "Себестоимость продаж, тыс.руб.") OVER(PARTITION BY "Месяц") AS "Маржа всего за месяц, тыс. руб.",
        SUM("Сумма продаж, тыс.руб." - "Себестоимость продаж, тыс.руб.") OVER(PARTITION BY "Номенклатура", "Месяц")/
        SUM("Сумма продаж, тыс.руб." - "Себестоимость продаж, тыс.руб.") OVER(PARTITION BY "Месяц") AS "Доля маржи"
    FROM
        df_test
    ORDER BY
        "Месяц", "Доля маржи" DESC, "Номенклатура"
"""

df_Source_Calculations = duckdb.sql(query_sql).df()
df_Source_Calculations

,"""Формат магазина",Магазин,Группа,Номенклатура,Месяц,"Кол проданного товара, шт","Сумма продаж, тыс.руб.","Себестоимость продаж, тыс.руб.","Остаток на конец дня среднедневной, шт","Себестоимость остатков на конец дня среднедневная, тыс.руб.""","Цена за ед., тыс.руб","Себес. за ед., тыс. руб.","Маржа за ед., тыс. руб","Маржа всего за прод, тыс. руб.","Маржа всего за месяц, тыс. руб.",Доля маржи
0,"""Супер",Магазин 14,Овощи и фрукты,Овощи и фрукты 255,7,"1,034.81",28.80,17.40,33.60,0.41,0.03,0.02,0.01,243.70,"15,112.20",0.02
1,"""Супер",Магазин 22,Овощи и фрукты,Овощи и фрукты 255,7,"1,860.72",51.61,32.38,328.43,4.76,0.03,0.02,0.01,243.70,"15,112.20",0.02
2,"""Супер",Магазин 31,Овощи и фрукты,Овощи и фрукты 255,7,958.62,26.96,16.71,80.34,0.96,0.03,0.02,0.01,243.70,"15,112.20",0.02
3,"""Супер",Магазин 25,Овощи и фрукты,Овощи и фрукты 255,7,910.98,27.58,15.97,88.01,1.06,0.03,0.02,0.01,243.70,"15,112.20",0.02
4,"""Гипер",Магазин 8,Овощи и фрукты,Овощи и фрукты 255,7,"2,935.09",82.04,48.13,122.79,1.46,0.03,0.02,0.01,243.70,"15,112.20",0.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49898,"""Супер",Магазин 31,Овощи и фрукты,Овощи и фрукты 323,9,"1,233.96",65.83,70.43,62.01,3.93,0.05,0.06,-0.00,-88.24,"13,123.91",-0.01
49899,"""Супер",Магазин 25,Овощи и фрукты,Овощи и фрукты 323,9,"1,265.93",71.86,70.44,81.23,5.15,0.06,0.06,0.00,-88.24,"13,123.91",-0.01
49900,"""Супер",Магазин 20,Овощи и фрукты,Овощи и фрукты 323,9,837.61,47.43,44.98,118.15,5.67,0.06,0.05,0.00,-88.24,"13,123.91",-0.01
49901,"""Гипер",Магазин 8,Овощи и фрукты,Овощи и фрукты 323,9,"2,575.83",138.94,143.52,151.57,7.55,0.05,0.06,-0.00,-88.24,"13,123.91",-0.01


# ***Negative_Margin***

In [145]:

query_sql3 = """
    WITH t1 AS (
    SELECT
        "Номенклатура",
        SUM("Кол проданного товара, шт") AS "Продано, шт",
        SUM("Маржа всего за прод, тыс. руб.") AS "Маржа, тыс.руб."
    FROM df_Source_Calculations
    WHERE "Месяц" = 9
    GROUP BY "Номенклатура"
    ORDER BY "Маржа, тыс.руб." DESC
    )
        SELECT
            *,
            9 AS "Месяц"
        FROM
            t1
        WHERE
            "Маржа, тыс.руб." < 0
        ORDER BY
            "Маржа, тыс.руб." ASC

"""

df_Negative_Margin = duckdb.sql(query_sql3).df()
df_Negative_Margin

,Номенклатура,"Продано, шт","Маржа, тыс.руб.",Месяц
0,Овощи и фрукты 323,"22,977.36","-1,058.87",9
1,Мясо и мясопродукты 340,"2,040.00",-207.71,9
2,Мясо и мясопродукты 326,375.30,-87.46,9
3,Мясо и мясопродукты 324,901.70,-84.82,9
4,Мясо и мясопродукты 39,"1,075.43",-75.63,9
5,Мясо и мясопродукты 478,"13,378.11",-68.14,9
6,Мясо и мясопродукты 480,"3,659.04",-53.97,9
7,Овощи и фрукты 144,201.00,-31.48,9
8,Мясо и мясопродукты 325,342.88,-27.07,9
9,Овощи и фрукты 347,63.34,-5.48,9


# ***ABC_Analysis***

In [146]:

query_sql2 = """

With t1 AS
    (
    SELECT
        "Номенклатура",
        SUM("Кол проданного товара, шт") AS "Продано, шт",
        SUM("Маржа всего за прод, тыс. руб.") AS "Маржа, тыс.руб."
    FROM df_Source_Calculations
    WHERE "Месяц" = 9
    GROUP BY "Номенклатура"
    ORDER BY "Маржа, тыс.руб." DESC
    ),
        t2  AS (
            SELECT
                *,
                SUM("Маржа, тыс.руб.") OVER() AS "Суммарная маржа, тыс. руб.",
                "Маржа, тыс.руб."/ SUM("Маржа, тыс.руб.") OVER() AS "Доля Маржи"
            FROM
                t1
            WHERE "Маржа, тыс.руб." > 0
                )
                  SELECT
                      *,
                      SUM("Доля Маржи") OVER(ORDER BY "Доля Маржи" DESC) AS "Маржа нараст",
                      CASE
                          WHEN (SUM("Доля Маржи") OVER(ORDER BY "Доля Маржи" DESC)) <= 0.8 THEN 'A'
                          WHEN (SUM("Доля Маржи") OVER(ORDER BY "Доля Маржи" DESC)) <= 0.95 THEN 'B'
                          ELSE 'C'
                      END AS "ABC_Категория"
                  FROM
                      t2
                      ORDER BY
                      "Доля Маржи" DESC

"""
df_ABC_Analysis = duckdb.sql(query_sql2).df()
df_ABC_Analysis


,Номенклатура,"Продано, шт","Маржа, тыс.руб.","Суммарная маржа, тыс. руб.",Доля Маржи,Маржа нараст,ABC_Категория
0,Молочные продукты 612,"1,292.02","1,517.29","142,856.66",0.01,0.01,A
1,Овощи и фрукты 245,"37,941.42","1,488.37","142,856.66",0.01,0.02,A
2,Хлеб и хлебобулочные изд 26,"8,451.00","1,423.00","142,856.66",0.01,0.03,A
3,Овощи и фрукты 259,"14,012.83","1,322.96","142,856.66",0.01,0.04,A
4,Овощи и фрукты 355,"16,868.54","1,234.82","142,856.66",0.01,0.05,A
...,...,...,...,...,...,...,...
1894,Овощи и фрукты 274,1.40,0.01,"142,856.66",0.00,1.00,C
1895,Овощи и фрукты 348,7.35,0.01,"142,856.66",0.00,1.00,C
1896,Производство 44,1.00,0.01,"142,856.66",0.00,1.00,C
1897,Овощи и фрукты 269,2.55,0.01,"142,856.66",0.00,1.00,C


# ***Turnover_Comparison***

In [147]:
query_sql4 = """

     WITH t1 AS
     (
    SELECT "Номенклатура"
    FROM df_ABC_Analysis
    WHERE "ABC_Категория" = 'A'
      ),
        t2 AS (
            SELECT
                s."Номенклатура",
                s."Месяц",
                'A' AS "ABC_Категория",
                SUM(s."Кол проданного товара, шт") AS "Продано",
                AVG(s."Остаток на конец дня среднедневной, шт") AS "Средний остаток",
                SUM(s."Кол проданного товара, шт") /NULLIF(AVG(s."Остаток на конец дня среднедневной, шт"), 0) AS "Оборачиваемость"
            FROM df_Source_Calculations s
            JOIN t1
                ON s."Номенклатура" = t1."Номенклатура"
            WHERE s."Месяц" IN (8,9)
            GROUP BY
                s."Номенклатура",
                s."Месяц"
              ),
              t3 AS (
                      SELECT
                          'A' AS "ABC_Категория",
                          "Номенклатура",
                          COALESCE(MAX(CASE WHEN "Месяц" = 8 THEN "Оборачиваемость" END), 0) AS "Оборачиваемость_Август",
                          COALESCE(MAX(CASE WHEN "Месяц" = 9 THEN "Оборачиваемость" END), 0) AS "Оборачиваемость_Сентябрь",
                      FROM
                          t2
                      GROUP BY
                          "Номенклатура"
                      )
                      SELECT
                          *,
                          "Оборачиваемость_Сентябрь" - "Оборачиваемость_Август" AS "Изменение",
                          CASE
                              WHEN ("Оборачиваемость_Сентябрь" - "Оборачиваемость_Август") > 0 THEN 'Нет'
                              ELSE 'Да'
                          END AS "Ухудшение"
                      FROM t3


"""

df_Turnover_Comparison = duckdb.sql(query_sql4).df()
df_Turnover_Comparison

,ABC_Категория,Номенклатура,Оборачиваемость_Август,Оборачиваемость_Сентябрь,Изменение,Ухудшение
0,A,Овощи и фрукты 245,94.43,132.81,38.38,Нет
1,A,Мясо и мясопродукты 435,155.43,133.93,-21.50,Да
2,A,Молочные продукты 237,103.50,117.01,13.51,Нет
3,A,Овощи и фрукты 250,138.08,126.02,-12.06,Да
4,A,Мясо и мясопродукты 218,96.28,24.39,-71.90,Да
...,...,...,...,...,...,...
540,A,Молочные продукты 596,28.26,21.51,-6.75,Да
541,A,Производство 35,15.27,49.97,34.70,Нет
542,A,Овощи и фрукты 241,0.00,24.47,24.47,Нет
543,A,Овощи и фрукты 210,0.00,22.58,22.58,Нет


# ***Calculate the return on sales***

In [148]:
query_sq5 = """

      SELECT
          "Номенклатура",
          SUM("Сумма продаж, тыс.руб.") AS "Выручка",
          SUM("Сумма продаж, тыс.руб." - "Себестоимость продаж, тыс.руб.") AS "Маржа",
          (SUM("Сумма продаж, тыс.руб." - "Себестоимость продаж, тыс.руб.")) / NULLIF(SUM("Сумма продаж, тыс.руб."),0) AS "Рентабельность"
      FROM df_Source_Calculations
      GROUP BY "Номенклатура"
      ORDER BY "Рентабельность" DESC
      LIMIT 5

"""

df_return_on_sale = duckdb.sql(query_sq5).df()
df_return_on_sale

,Номенклатура,Выручка,Маржа,Рентабельность
0,Овощи и фрукты 301,0.81,0.74,0.91
1,Овощи и фрукты 169,1.92,1.27,0.66
2,Овощи и фрукты 121,2.59,1.65,0.64
3,Овощи и фрукты 329,0.73,0.46,0.64
4,Овощи и фрукты 181,3.46,2.18,0.63


In [149]:

with pd.ExcelWriter('/content/drive/MyDrive/Проекты_Санег/Коды_только/solved_test.xlsx') as writer:
    df_Source_Calculations.to_excel(writer, sheet_name='Source_Calculations')
    df_Negative_Margin.to_excel(writer, sheet_name='Negative_Margin')
    df_ABC_Analysis.to_excel(writer, sheet_name='ABC_Analysis')
    df_Turnover_Comparison.to_excel(writer, sheet_name='Turnover_Comparison')
    df_return_on_sale.to_excel(writer, sheet_name='return_on_sale')



Exception ignored in: <function ZipFile.__del__ at 0x7a12dbd5d300>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file
